LIBRERIA FACE_RECOGNITION: IDENTITA' BIOMETRICA ED EMBEDDINGS

Sappiamo mappere un volte ed indivisuarne le caratteristiche, oggi vedremo come rispondere alla domanda: "Chi è questa persona?"

Non vediamo più i pixel come semplice colori, ma come coordinate di una identità

- Face Embedding: trasformare i tratti somatici in un vettore da 128 dimensioni
- Il confronto da volti
- Database locale capace di ricordare e riconoscere persone note.

La libreria face_recognition è una libreria Python costruita sopra dlib che semplifica molto il riconoscimento facciale. Il concetto centrale non è "confrontare due foto", ma trasformare ogni volto in un vettore numerico e confrontare quei vettori.
immagine -> allinamento/landmarks -> rete neurale -> embdedding del volto -> confronto tra embedding -> stessa persona/persona diversa
Il volto viene proietttato in uno spazio multidimensionale dove la posizione del punto rappresenta l'identigà unica del soggetto, indipendentemente dall'illuminazione o dalla posa.

Le 128 Dimensioni dell'Identità
Il cuore vero è l'embedding

Cos'è l'embedding facciale
La libreria prende un volto e lo converte in un vettore di 128 numeri.
Esempio semplificato:
foto Barbara -> [0.12,-0.24,0.55,0.08,...]
Nella realtà ci sono 128 valori.
La documentazione di dlib descrive proprio un modello che mappa il volta in uno spazio euclideo a 128 dimensioni, dove volti della stessa persona tendono a risultare vicini.
Non sono però 128 numeri casuali, ognuno di essi rappresenta una caratteristica di alto livello come la curvatura del mento o la distanza tra le sopracciglia.
Invece di gestire milioni di pixel riduciamo tutto a un vettore di numeri.

Quindi non memorizza: "questa è Barbara"
Impara piuttosto una funzione che dalla foto di un volto trova questi 128 valori.
Poi siamo noi che dobbiamo collegare quel vettore ad un'identità
Questo è il senso dell'IDENTIA' BIOMETRICA
Il dato biometrico non è il nome, è la rappresentazione matematica del volto.
import face_recognition
image = face_recognition.load_image_file("barbara.jpg")
encoding = face_recognition.face_encodings(image)[0]  #genero encoding

La libreria face_recognition usa questi "face encoding" per confrontare volti
Dietro questa magia c'è la libreria Dlib

Il Ruolo della Rete Dlib
La libreria internamente utilizza un modello derivato da ResNet, addestrato su milioni di immagini per mappare i tratti facciali con estrema precisione.

Come riconosce la stessa persona?
i valori di encoding di foto diverse ma della stessa persona saranno geometricamente vicini, mentre i vettori di encoding di una persona diversa saranno geometricamente lontani dalla prima persona.
Questa è esattamente la logica del Deep Metric Learning

face_encodings()
questa è la funzione che riceve l'immagnie (già allineata tramite i landmarks) e restituisce il vettore numerico

face_distance()
la libreria ti permette di vedere direttamente la distanza tra due vettori di volti
distance = face_recognition.face_distance( [encoding_barbara],encoding_nuovo)
print(distance)
potesti ottenere 0.32 oppure 0.81
qui devi ricordare:
distanza piccola -> volti simili
distanza grande -> volti diversi

compare_face()
con l'istruzione:
result = face_recognition.compare_faces([encoding_barbara],encoding_nuovo)
potesti ottenere: True oppure False
true= distanza entro la soglia
false= distanza oltre la soglia
La libreria usa normalmente una tolerance predefinita di 0.6, valori più bassi rendono il confronto più servero.

Ma come ha fatto la rete ad imparare quali numeri assegnare a chi?

La Triplet Loss e l'Ottimizzazione
Come la rete impara le distanze
Durante l'addestramento, il modello utilzza una funzoine di perdita chiamata Triplet Loss.
Questa funzione considera tre immagini:
un'ancora, una positiva (stessa persona) e una negativa (persona diversa)
Diciamo alla rete: sposta il positivo più vicino all'ancora e allontana il negativo.
L'obbiettivo è minimizzare la distanza tra l'ancora e il positivo garantendo al contempo che il negativo sia separato da un margine minimo definito.
Alla fine questo spostamento crea dei cluster, dei punti vicini per ogni identità

Ora che abbiamo i punti, come misuriamo quanto sono vicini?

Confronta tramite Distanza Euclidea
Misurare la somiglianza matematica
Una volta ottenuti i vettori a 128 dimensioni per due volti diversi, il riconoscimento diventa un problema di geometria analitica. Se i due vettori sono vicini nello spazio multidimensionale, appartengono con alta probabilità alla stessa persona.
La distanza euclidea rappresenta la linea retta tra due punti in questo spazio. Definendo una soglia critica (threshold), possiamo decidere se il sistema deve confrmare o rifiutare l'identità

Ma dobbiamo decidere quando un vicino è considerato abbastanza vicino da essere considerato un match
Dobbiamo quindi impostare una soglia di tolleranza

Soglie e Tolleranza
Il bilanciamento tra sicurezza e usabilità
- Distanza Lineare: il calcolo della radice quadrata della somma dei quadrati delle differenze tra i componenti dei due vettori
- Default Threshold: nella libreria face_recognition, il valore standard è spesso 0.6. Valori inferiori indicano una somiglianza maggiore
- False Acceptance Rate (FAR): se la soglia è troppo alta, il sistema potrebbe confondere due persone diverse (rischio sicurezza)
- False Rejection Rate (FRR): se la soglia è troppo bassa, il sistema potrebbe non riconoscere l'utente legittimo (probelam di UX)

Normalmente si utilizza la soglia di 0.6 ma per sistemi ad alta sicurezza si può scendere a 0.4

La libreria ci offre già tutti gli strumenti pronti per gestire questi calcoli con le funzioni che abbiamo visto prima: compare_faces(), ci da un verdetto secco si o no - face_distance(), restituisce un valore numerico della distanza - ed altre

Il calcolo della distanza è implementato tramite funzioni Numpy, permettendo il confronto di un volto con migliia di altri quasi istantaneamente.

Ma dove salviamo questi 128 numeri senza doverli ricalcolare ogni volta?
dobbiamo creare un databse locale

Creazione di un Database Locale
Archiviazione e Recupero delle Identità
Per costruire un sistema di riconoscimento funzionante, non possiamo limitarci a processare le immagini in tempo reale. Dobbiamo creare una memoria storica: un database che associ i nomi o ID ai rispettivi embeddings.
Vediamo come organizzare i file locali, estrarre le codifiche una sola volta e salvarle per utilizzi futuri senza dover ricalcolare la CNN ad ogni avvio.

vediamo come oraganizzare questa memoria digitale

Gestione delle Identità Notate
Dall'immagine al record persistente.
- Struggura Folder-based: Organizza le immagini in cartelle nominate con il nome del soggetto per facilitare il caricamento batch
- Persistenza dei Dati: salvare gli array di embeddings in formati come Pickle o JSON per evitare l'overhead del ricalcolo
- Associazione Nome-Vettore: mantenere due liste parallele o un dizionario che mappi ogni indice con encoding a una stringa identificativa
- Scaling: per database con migliaia di volti, l'approccio lineare può rallentare, in quei casi si ricorre a librerie di ricerca vettoriale come FAISS


C'è però un aspetto logale da non sottovalutare
Il wordflos di validazione deve essere pulito

Workflow di Registrazione
- Caricamento immagine: utilizzo load_image_file() per convertire il file su disco in array numpy 
- Validazione in Fase di Encode: varificare sempre che sia presente esattamente un volto nell'immagine di registrazione per evitare ambiguità nel database 
- Sicurezza e Privacy: gli embeddings sono dati biometrici. Sebbene non sia possibile ricostruire facilmente un volto originale dal vettore, la loro protezione è soggetta a normative come il GDPR
Non possiamo conservare questi dati senza consenso e protezione adeguata.

Ma una volta che il database è pronto, come troviamo il vincitore tra mille sospetti

Ottimizzazione del Match Lineare
Ricerca del Best Match
Quando confrontiamo un volto sconosciuto con il database, otteniamo una lista di distanza. Il Best Match è rappresentato dall'indice che minimizza questa distanza, a patto che sia sotto la soglia di sicurezza.
L'operazione argmin di numpy ci permette di identificare istantaneamente il record più simile tra centianio di profili archiviati.

In [ ]:
"""
SISTEMA DI RICONOSCIMENTO FACCIALE 
----------------------------------------------------------------------
Questo script implementa un pipeline riconoscimento biometrico.
Il sistema non si limita a confrontare immagini (pixel-by-pixel), ma estrae
una rappresentazione matematica astratta del volto.

COMPONENTI LOGICI:
1. INPUT: Caricamento immagini tramite 'face_recognition'.
2. EMBEDDING: Trasformazione del volto in un vettore numerico (128-d).
3. DATABASE: Archiviazione persistente dei vettori tramite 'pickle'.
4. INFERENZA: Confronto tra vettori tramite Distanza Euclidea.
"""

import os
import numpy as np
import pickle
from pathlib import Path
from typing import Optional, Dict, Tuple, List

# ==========================================================
# 1. SEZIONE CONFIGURAZIONE
# ==========================================================
# BASE_DIR identifica dinamicamente la cartella dove si trova questo script.
# Questo garantisce che il programma trovi le immagini se salvate nella stessa cartella.
BASE_DIR = Path(__file__).resolve().parent

# PATH_DATABASE: Percorso del file binario dove verranno archiviati i volti noti.
PATH_DATABASE = BASE_DIR / "face_vault_2026.pkl"

# NOME_SOGGETTO: Etichetta testuale assegnata alla persona durante la registrazione.
NOME_SOGGETTO = "Elon Musk"

# PATH_REGISTRAZIONE: L'immagine sorgente usata come riferimento nel database.
PATH_REGISTRAZIONE = BASE_DIR / "download.jpg"

# PATH_TEST: L'immagine 'ignota' che il sistema deve provare a riconoscere.
PATH_TEST = BASE_DIR / "download (1).jpg"
# ==========================================================

# Configurazione Backend: Keras 3 è l'interfaccia standard moderna.
# Impostiamo PyTorch come motore di calcolo per ottimizzare le operazioni sui tensori.
os.environ["KERAS_BACKEND"] = "torch"
import keras
import face_recognition

class FaceIDSystem2026:
    """
    Agisce come 'Orchestratore' del sistema.
    Gestisce l'interazione tra i file su disco (Database .pkl) e 
    la memoria RAM (self.database), coordinando i modelli di Computer Vision.
    """
    
    def __init__(self, db_path: Path):
        """
        Costruttore del sistema. Inizializza il database e verifica il backend.
        
        Args:
            db_path (Path): Oggetto Path che punta al file del database .pkl
        """
        self.db_path = db_path
        # All'avvio, carichiamo subito i dati dal disco alla RAM (dizionario).
        # Questo permette di fare confronti istantanei senza leggere ogni volta il file.
        self.database: Dict[str, np.ndarray] = self._load_database()
        
        print(f"--- LOG: Sistema Inizializzato (Backend: {keras.backend.backend()}) ---")

    def _load_database(self) -> Dict:
        """
        Legge il file .pkl e ricostruisce l'oggetto Python originario.
        Interazione: RAM <--- DISCO (Deserializzazione)
        """
        if self.db_path.exists():
            with open(self.db_path, 'rb') as f:
                try:
                    # Carica il dizionario degli embeddings già salvati.
                    data = pickle.load(f)
                    return data if isinstance(data, dict) else {}
                except Exception as e:
                    print(f"Errore caricamento: {e}")
                    return {}
        return {}

    def _load_image(self, path: Path) -> Optional[np.ndarray]:
        """
        Carica un file immagine e lo converte in una matrice di pixel RGB.
        
        Args:
            path (Path): Percorso del file immagine.
            
        Returns:
            Optional[np.ndarray]: Array NumPy dei pixel o None se il file manca.
        """
        if not path.exists():
            print(f"Errore: Il file '{path.name}' non è stato trovato in {path.parent}")
            return None
        try:
            # Utilizza la utility di face_recognition per garantire il formato corretto.
            return face_recognition.load_image_file(str(path))
        except Exception as e:
            print(f"Errore nel caricamento dell'immagine {path.name}: {e}")
            return None

    def _get_embedding(self, image_array: np.ndarray) -> Optional[np.ndarray]:
        """
        IL MOTORE AI: Estrae le 'Feature' (Caratteristiche).
        Questa riga invoca una ResNet pre-addestrata che:
        1. Trova il volto nell'immagine.
        2. Allinea il volto (ruota occhi/menton per averlo dritto).
        3. Genera 128 numeri che descrivono le distanze spaziali tra i tratti somatici.
        """
        # encodings sarà una lista di vettori (uno per ogni faccia trovata).
        encodings = face_recognition.face_encodings(image_array)
        
        # Restituiamo solo il primo volto trovato (indice 0).
        return encodings[0] if encodings else None

    def add_identity(self, name: str, image_path: Path):
        """
        Registra una nuova persona estraendo il suo embedding e salvandolo nel vault.
        """
        if name in self.database:
            print(f"Identità '{name}' già presente nel database.")
            return

        print(f"Registrazione in corso: {name}...")
        img_array = self._load_image(image_path)
        
        if img_array is not None:
            embedding = self._get_embedding(img_array)
            if embedding is not None:
                # Archiviazione dell'embedding (128 numeri) invece dell'intera immagine.
                self.database[name] = embedding
                self.save_db()
                print(f"OK: {name} registrato con successo nel vault.")
            else:
                print(f"Errore: Nessun volto rilevato in {image_path.name}.")

    def save_db(self):
        """
        Persiste il dizionario degli embeddings su disco in formato binario.
        """
        with open(self.db_path, 'wb') as f:
            pickle.dump(self.database, f)

    def identify(self, image_path: Path, threshold: float = 0.5) -> Tuple[str, float]:
        """
        Fase di Inferenza (Confronto):
        Confronta il volto ignoto con TUTTI i volti nel vault.
        
        Logica Matematica:
        Viene usata la 'Distanza Euclidea'. Se la distanza è 0, i volti sono identici.
        Più la distanza cresce, meno i due volti sono simili.
        """
        # 1. Carichiamo l'immagine di test
        test_img = self._load_image(image_path)
        if test_img is None: return "File Invalido", 0.0

        # 2. Generiamo l'embedding del volto da identificare
        unknown_encoding = self._get_embedding(test_img)
        if unknown_encoding is None: return "Nessun Volto Trovato", 0.0

        # 3. Prepariamo i dati del vault per il confronto massivo
        names = list(self.database.keys())
        known_encodings = np.array(list(self.database.values()))
        
        if not names: return "Database Vuoto", 0.0

        # 4. CALCOLO DELLE DISTANZE (Linear Algebra Optimization)
        # face_distance calcola la distanza tra il vettore ignoto e TUTTI quelli del DB contemporaneamente.
        distances = face_recognition.face_distance(known_encodings, unknown_encoding)
        
        # 5. np.argmin trova l'indice del numero più piccolo nel vettore 'distances'.
        # Quel numero rappresenta il nostro 'Best Match'.
        best_match_idx = np.argmin(distances)
        
        # Trasformiamo la distanza (0.0-1.0) in una percentuale di confidenza (es. 0.2 dist -> 80% conf).
        confidence = 1 - distances[best_match_idx]

        # 6. Verifica della soglia (Tolerance)
        # Se anche la distanza più bassa è troppo alta, diciamo "Sconosciuto".
        if distances[best_match_idx] <= threshold:
            return names[best_match_idx], confidence
            
        return "Sconosciuto", confidence

def main():
    """
    Punto di ingresso dello script. Coordina le fasi di registrazione e test.
    """
    # STEP 0: Inizializzazione
    face_system = FaceIDSystem2026(PATH_DATABASE)

    # --- FASE 1: REGISTRAZIONE ---
    # Se la foto di registrazione esiste e il soggetto non è nel database, procediamo.
    if PATH_REGISTRAZIONE.exists():
        face_system.add_identity(NOME_SOGGETTO, PATH_REGISTRAZIONE)
    else:
        print(f"Nota: Foto registrazione '{PATH_REGISTRAZIONE.name}' non trovata. Controllo database esistente...")

    # --- FASE 2: RICONOSCIMENTO ---
    print("\n" + "="*45)
    print("      SESSIONE RICONOSCIMENTO LOCALE 2026")
    print("="*45)

    if PATH_TEST.exists():
        # Eseguiamo l'inferenza sull'immagine di test.
        nome, conf = face_system.identify(PATH_TEST)
        print(f"SOGGETTO RILEVATO: {nome}")
        print(f"LIVELLO CONFIDENZA: {conf:.2%}")
        
        # Messaggio di feedback basato sull'esito.
        if nome != "Sconosciuto" and nome != "Nessun Volto":
            print(f"Esito: Identificazione di '{nome}' completata con successo.")
        else:
            print("Esito: Soggetto non presente nel vault o volto non chiaro.")
    else:
        print(f"Errore: File di test '{PATH_TEST.name}' non trovato.")

    print("="*45)

if __name__ == "__main__":
    main()